In [1]:
import pandas as pd
import numpy as np
import os

# =========================================================
# Given 10-year significant wave height
# =========================================================
Hs_10 = 7.559

lower = Hs_10 * 0.9
upper = Hs_10 * 1.1

print(f"Filtering range for WVHT: {lower:.3f} m to {upper:.3f} m")

# =========================================================
# Invalid values used in NDBC files
# =========================================================
invalid_vals = [99, 99.0, 999, 999.0]

# =========================================================
# File paths
# =========================================================
files = [
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2011.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2012.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2013.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2014.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2015.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2016.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2017.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2018.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2019.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2020.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2021.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2022.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2023.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2024.txt",
    r"D:\Dipen Saha\Course Work\Spring 2026\Homework\Homework-5\wave_data_qn-5\2025.txt"
]

# =========================================================
# Check existing files only
# =========================================================
existing_files = [f for f in files if os.path.exists(f)]

if not existing_files:
    raise FileNotFoundError("No input files were found. Check the file paths.")

print(f"Number of files found: {len(existing_files)}")

# =========================================================
# Collect filtered data
# =========================================================
filtered_data = []

for file in existing_files:
    print(f"Reading: {os.path.basename(file)}")

    df = pd.read_csv(
        file,
        delim_whitespace=True,
        skiprows=[1],
        engine="python"
    )

    # Clean column names
    df.columns = df.columns.str.strip()

    # Convert required columns to numeric safely
    required_cols = ["WVHT", "DPD", "MWD"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in file: {file}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove invalid values and NaNs
    df = df[
        (~df["WVHT"].isin(invalid_vals)) &
        (~df["DPD"].isin(invalid_vals)) &
        (~df["MWD"].isin(invalid_vals)) &
        (df["WVHT"].notna()) &
        (df["DPD"].notna()) &
        (df["MWD"].notna())
    ]

    # Filter by ±10% of 10-year wave height
    df_filtered = df[(df["WVHT"] >= lower) & (df["WVHT"] <= upper)]

    if not df_filtered.empty:
        filtered_data.append(df_filtered)

# =========================================================
# Combine filtered records
# =========================================================
if not filtered_data:
    raise ValueError("No records found within the specified wave height range.")

filtered_df = pd.concat(filtered_data, ignore_index=True)

print("\nNumber of extreme wave records:", len(filtered_df))

# =========================================================
# Range of dominant wave period and direction
# =========================================================
dpd_min = filtered_df["DPD"].min()
dpd_max = filtered_df["DPD"].max()

mwd_min = filtered_df["MWD"].min()
mwd_max = filtered_df["MWD"].max()

print("\nDominant Wave Period (DPD) Range:")
print(f"{dpd_min:.2f} s to {dpd_max:.2f} s")

print("\nWave Direction (MWD) Range:")
print(f"{mwd_min:.0f}° to {mwd_max:.0f}°")

# =========================================================
# Average dominant wave period
# =========================================================
dpd_mean = filtered_df["DPD"].mean()

print("\nAverage Dominant Wave Period (DPD):")
print(f"{dpd_mean:.2f} s")

# =========================================================
# Circular mean of wave direction
# =========================================================
mwd_rad = np.deg2rad(filtered_df["MWD"])

sin_mean = np.mean(np.sin(mwd_rad))
cos_mean = np.mean(np.cos(mwd_rad))

mwd_circular = np.rad2deg(np.arctan2(sin_mean, cos_mean))

if mwd_circular < 0:
    mwd_circular += 360

print("\nAverage Wave Direction (MWD, circular mean):")
print(f"{mwd_circular:.2f}°")

# =========================================================
# Optional summary
# =========================================================
print("\nSummary:")
print(f"Filtered WVHT range used: {lower:.3f} m to {upper:.3f} m")
print(f"Average DPD: {dpd_mean:.2f} s")
print(f"Average MWD: {mwd_circular:.2f}°")

Filtering range for WVHT: 6.803 m to 8.315 m
Number of files found: 15
Reading: 2011.txt
Reading: 2012.txt


C:\Users\sahad2\AppData\Local\Temp\ipykernel_19492\3459237267.py:59: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
C:\Users\sahad2\AppData\Local\Temp\ipykernel_19492\3459237267.py:59: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


Reading: 2013.txt


C:\Users\sahad2\AppData\Local\Temp\ipykernel_19492\3459237267.py:59: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


Reading: 2014.txt


...

Reading: 2024.txt


C:\Users\sahad2\AppData\Local\Temp\ipykernel_19492\3459237267.py:59: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


Reading: 2025.txt


C:\Users\sahad2\AppData\Local\Temp\ipykernel_19492\3459237267.py:59: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(



Number of extreme wave records: 17

Dominant Wave Period (DPD) Range:
10.81 s to 13.79 s

Wave Direction (MWD) Range:
23° to 76°

Average Dominant Wave Period (DPD):
11.87 s

Average Wave Direction (MWD, circular mean):
48.37°

Summary:
Filtered WVHT range used: 6.803 m to 8.315 m
Average DPD: 11.87 s
Average MWD: 48.37°


In [5]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import root_scalar

# =========================================================
# GIVEN DATA
# =========================================================
rho = 1000.0          # kg/m^3
g = 9.81              # m/s^2
CD = 1.5
bv = 0.007            # m (7 mm)
hv = 0.6              # m vegetation height
a = 1.008             # storm surge amplitude (m)
T = 6 * 3600          # 6 hours in seconds
sigma = 2 * np.pi / T

# ---------------------------------------------------------
# IMPORTANT:
# Option 1: use regular-wave height H = 2a = 2.016 m
# Option 2: if your instructor intends direct use of surge
#           amplitude in the vegetation formula, use Hs = a
# ---------------------------------------------------------
#Hs = 2 * a   # try this first
Hs = a     # uncomment this instead if needed

# =========================================================
# TARGET ENERGY TO DISSIPATE (50% of incoming energy)
# =========================================================
E0 = (1/8) * rho * g * (2 * a)**2   # incoming regular-wave energy from part (a)
E_target = 0.5 * E0

print("=" * 60)
print(f"Using Hs = {Hs:.3f} m in the vegetation dissipation formula")
print(f"Incoming energy E0 = {E0:.6f} J/m^2")
print(f"Target dissipation = {E_target:.6f} J/m^2")
print("=" * 60)

# =========================================================
# DISPERSION RELATION SOLVER
# sigma^2 = g k tanh(kh)
# =========================================================
def solve_k(h):
    if h <= 0:
        raise ValueError("Water depth must be positive.")

    # shallow-water initial guess
    k = sigma / np.sqrt(g * h)

    for _ in range(100):
        f = g * k * np.tanh(k * h) - sigma**2
        df = g * np.tanh(k * h) + g * k * h * (1 / np.cosh(k * h))**2
        k_new = k - f / df

        if abs(k_new - k) < 1e-14:
            return k_new
        k = k_new

    return k

# =========================================================
# GROUP VELOCITY
# =========================================================
def group_velocity(h):
    k = solve_k(h)
    C = sigma / k
    n = 0.5 * (1 + (2 * k * h) / np.sinh(2 * k * h))
    Cg = n * C
    return Cg, k

# =========================================================
# WATER DEPTH ACROSS THE MARSH
# x = 0 at shoreline (seaward edge)
# x = W at landward edge
#
# Landward edge elevation = design still water level
# Surge amplitude above still water level = a
# So minimum water depth over marsh during surge crest = a
# =========================================================
def h_of_x(x, W):
    return a + (W - x) / 30.0

# =========================================================
# DISSIPATION INTEGRAND
# =========================================================
def integrand(x, W, Nv):
    h = h_of_x(x, W)

    Cg, k = group_velocity(h)

    term1 = 1 / (2 * np.sqrt(np.pi))
    term2 = Hs**3
    term3 = rho * CD * bv * Nv
    term4 = ((k * g) / (2 * sigma))**3
    term5 = (np.sinh(k * hv)**3 + 3 * np.sinh(k * hv)) / (3 * k * np.cosh(k * h)**3)

    return (1 / Cg) * term1 * term2 * term3 * term4 * term5

# =========================================================
# TOTAL DISSIPATED ENERGY OVER MARSH WIDTH W
# =========================================================
def E_diss(W, Nv):
    if W <= 0:
        return 0.0

    val, _ = quad(integrand, 0, W, args=(W, Nv), limit=500)
    return val

# =========================================================
# DIAGNOSTIC PRINT
# =========================================================
def diagnostic_check():
    print("\nDIAGNOSTIC CHECK")
    print("-" * 60)
    test_widths = [1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
    for Nv in [300, 1200]:
        print(f"\nNv = {Nv} plants/m^2")
        for W in test_widths:
            val = E_diss(W, Nv)
            diff = val - E_target
            print(f"W = {W:8.4f} m | E_diss = {val:12.6f} | diff = {diff:12.6f}")
    print("-" * 60)

# =========================================================
# AUTOMATIC BRACKET SEARCH
# =========================================================
def find_bracket(Nv, W_start=1e-4, W_max=1e6, factor=2.0, verbose=True):
    W1 = W_start
    f1 = E_diss(W1, Nv) - E_target

    W2 = W1 * factor
    while W2 <= W_max:
        f2 = E_diss(W2, Nv) - E_target

        if verbose:
            print(f"Trying bracket: W1={W1:.6e}, f1={f1:.6f}; W2={W2:.6e}, f2={f2:.6f}")

        if f1 == 0:
            return W1, W1

        if f1 * f2 < 0:
            return W1, W2

        W1, f1 = W2, f2
        W2 *= factor

    raise ValueError("Could not find a bracket containing the root. Increase W_max or check Hs assumption.")

# =========================================================
# SOLVE FOR WIDTH
# =========================================================
def solve_width(Nv, verbose=True):
    W1, W2 = find_bracket(Nv, verbose=verbose)

    if W1 == W2:
        return W1

    func = lambda W: E_diss(W, Nv) - E_target
    sol = root_scalar(func, bracket=[W1, W2], method='brentq')

    return sol.root

# =========================================================
# MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    diagnostic_check()

    for Nv in [300, 1200]:
        print(f"\n{'='*60}")
        print(f"Solving for Nv = {Nv} plants/m^2")
        print(f"{'='*60}")

        try:
            W_sol = solve_width(Nv, verbose=True)
            E_sol = E_diss(W_sol, Nv)

            print(f"\nRequired marsh width W = {W_sol:.6f} m")
            print(f"Dissipated energy     = {E_sol:.6f} J/m^2")
            print(f"Target energy         = {E_target:.6f} J/m^2")

        except ValueError as e:
            print(f"\nCould not solve for Nv = {Nv}")
            print(str(e))

    print("\nDone.")

Using Hs = 1.008 m in the vegetation dissipation formula
Incoming energy E0 = 4983.793920 J/m^2
Target dissipation = 2491.896960 J/m^2

DIAGNOSTIC CHECK
------------------------------------------------------------

Nv = 300 plants/m^2
W =   0.0001 m | E_diss =     0.065901 | diff = -2491.831059
W =   0.0010 m | E_diss =     0.658995 | diff = -2491.237965
W =   0.0100 m | E_diss =     6.587988 | diff = -2485.308972
W =   0.0500 m | E_diss =    32.896443 | diff = -2459.000517
W =   0.1000 m | E_diss =    65.684460 | diff = -2426.212500
W =   0.5000 m | E_diss =   324.148750 | diff = -2167.748210
W =   1.0000 m | E_diss =   637.921420 | diff = -1853.975540
W =   5.0000 m | E_diss =  2827.563160 | diff =   335.666200
W =  10.0000 m | E_diss =  4952.451577 | diff =  2460.554617
W =  50.0000 m | E_diss = 12418.161206 | diff =  9926.264246
W = 100.0000 m | E_diss = 15301.493452 | diff = 12809.596492

Nv = 1200 plants/m^2
W =   0.0001 m | E_diss =     0.263606 | diff = -2491.633354
W =   0.001